# Hull Tactical Market Prediction - Submission

This notebook follows the official Kaggle evaluation API format.

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl

import kaggle_evaluation.default_inference_server

In [ ]:
# Configuration
MODEL_PATH_LGB = "/kaggle/input/hull-models/lgb_model.txt"  # Update with your actual dataset path
MODEL_PATH_XGB = "/kaggle/input/hull-models/xgb_model.json"
FEATURE_LIST_PATH = "/kaggle/input/hull-models/feature_list.json"

# Global model storage
MODELS = {}

In [ ]:
def load_models():
    """
    Load trained models. This can be called during the first predict() call
    since the first batch has no time limit.
    """
    global MODELS
    
    if MODELS:  # Already loaded
        return
    
    print("Loading models...")
    
    # Load LightGBM model
    try:
        if os.path.exists(MODEL_PATH_LGB):
            import lightgbm as lgb
            MODELS['lgb'] = lgb.Booster(model_file=MODEL_PATH_LGB)
            print("✓ LightGBM model loaded")
    except Exception as e:
        print(f"⚠ LightGBM model not loaded: {e}")
        MODELS['lgb'] = None
    
    # Load XGBoost model
    try:
        if os.path.exists(MODEL_PATH_XGB):
            import xgboost as xgb
            booster = xgb.Booster()
            booster.load_model(MODEL_PATH_XGB)
            MODELS['xgb'] = booster
            print("✓ XGBoost model loaded")
    except Exception as e:
        print(f"⚠ XGBoost model not loaded: {e}")
        MODELS['xgb'] = None
    
    # Load feature list
    try:
        if os.path.exists(FEATURE_LIST_PATH):
            import json
            with open(FEATURE_LIST_PATH, 'r') as f:
                MODELS['features'] = json.load(f)
            print(f"✓ Feature list loaded ({len(MODELS['features'])} features)")
    except Exception as e:
        print(f"⚠ Feature list not loaded: {e}")
        MODELS['features'] = None
    
    print("Model loading complete.")

In [ ]:
def build_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Feature engineering function.
    Replace this with your actual feature engineering logic.
    
    Args:
        df: Polars DataFrame with raw test data
    
    Returns:
        Polars DataFrame with engineered features
    """
    # Example: Select numeric columns only
    # Replace this with your actual feature engineering
    numeric_cols = [col for col in df.columns if df[col].dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]]
    
    features_df = df.select(numeric_cols)
    
    # Handle missing values
    features_df = features_df.fill_null(0.0)
    features_df = features_df.fill_nan(0.0)
    
    return features_df

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """
    Main prediction function called by Kaggle evaluation API.
    
    Args:
        test: Polars DataFrame containing a single timestep of test data
    
    Returns:
        float: Single prediction value (typically the target 'y')
    
    Notes:
        - First call has no time limit (for model loading)
        - Subsequent calls must return within 5 minutes
    """
    # Load models on first call (no time limit)
    if not MODELS:
        load_models()
    
    # Build features
    features = build_features(test)
    
    # Convert to numpy for model prediction
    X = features.to_numpy()
    
    # Get predictions from each model
    predictions = []
    
    # LightGBM prediction
    if MODELS.get('lgb') is not None:
        try:
            pred_lgb = MODELS['lgb'].predict(X)[0]
            predictions.append(pred_lgb)
        except Exception as e:
            print(f"LGB prediction error: {e}")
    
    # XGBoost prediction
    if MODELS.get('xgb') is not None:
        try:
            import xgboost as xgb
            dmat = xgb.DMatrix(X)
            pred_xgb = MODELS['xgb'].predict(dmat)[0]
            predictions.append(pred_xgb)
        except Exception as e:
            print(f"XGB prediction error: {e}")
    
    # Ensemble: average predictions
    if predictions:
        final_prediction = float(np.mean(predictions))
    else:
        # Fallback: return 0.0 if no models available
        final_prediction = 0.0
    
    return final_prediction

In [ ]:
# Initialize the Kaggle inference server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

# When running on Kaggle's hidden test set, serve the predictions
# When running locally, test with the public dataset
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    # Local testing mode
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))